<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB02 &mdash; Instrument Mastering</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Masters 308 instruments across seven asset classes into the LUSID security master, each with full identifiers and enrichment properties.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; <b>NB02</b> &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07</sub>

# NB02: Instrument Mastering

**What this does:** Loads 308 instruments across 7 asset classes from CSV files into the LUSID instrument master.

**Business context:** The instrument master (security master) is the foundation of any investment management platform. Every portfolio, transaction, and quote references instruments. EDM+ always starts with instruments because everything else depends on them.

**Asset classes loaded:**
| Type | Count | LUSID Model | Source File |
|------|-------|------------|-------------|
| Equities | 118 | Equity | common-stock.csv |
| Corporate Bonds | 61 | Bond | investment-grade.csv |
| Government Bonds | 27 | Bond | government.csv |
| ETFs | 75 | Equity | etfs.csv |
| Equity Options | 4 | EquityOption | equity-options.csv |
| FX Forwards | 4 | FxForward | fx-forwards.csv |
| Private Investments | 19 | SimpleInstrument | growth-equity.csv |

**Key patterns:** String properties must use `label_value` (not `metric_value`). Numeric properties (ESG scores, yields) use `metric_value`. FX Forwards require opposite signs for dom/fgn amounts. EquityOption type must be 'Vanilla'.

**Verify after running:** Go to Instruments in the LUSID web app. You should see 308 instruments.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Build APIs and Configuration

In [ ]:
instruments_api = lusid_factory.build(lu.InstrumentsApi)

SCOPE = 'edm-training2'
DATA_DIR = 'data/'

# Properties that LUSID stores as numbers (metric_value)
NUMERIC_PROPERTIES = {
    'DividendYield', 'MarketCapitalization', 'SharesOutstanding',
    'CouponRate', 'FaceValue', 'YieldToMaturity', 'AccruedInterest',
    'DefaultProbability', 'RecoveryRate', 'AUM', 'ExpenseRatio',
    'EsgEnvironmentRating', 'EsgSocialRating', 'EsgGovernanceRating',
}

def build_properties(row, prop_map, scope):
    props = {}
    for csv_col, prop_code in prop_map.items():
        val = row.get(csv_col)
        if pd.isna(val) or str(val).strip() == '': continue
        key = f'Instrument/{scope}/{prop_code}'
        if prop_code in NUMERIC_PROPERTIES:
            try:
                props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(
                    metric_value=lm.MetricValue(value=float(val), unit='')))
            except (ValueError, TypeError):
                props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(label_value=str(val).strip()))
        else:
            clean_val = str(val).strip()
            if clean_val.endswith('.0') and clean_val[:-2].isdigit():
                clean_val = clean_val[:-2]
            props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(label_value=clean_val))
    return props

def upsert_batch(definitions):
    try:
        resp = instruments_api.upsert_instruments(request_body=definitions, scope='default')
        ok = len(resp.values) if resp.values else 0
        fail = len(resp.failed) if resp.failed else 0
        if resp.failed:
            for k, v in list(resp.failed.items())[:3]:
                print(f'  FAILED: {k}: {str(v)[:200]}')
        return ok, fail
    except lu.ApiException as e:
        body = e.body[:300] if e.body else e.reason
        print(f'  ERROR: {body}')
        return 0, len(definitions)

print("✅ Helper functions ready")

---
## Load Equities

In [ ]:
print("--- Loading Equities ---")
df_eq = pd.read_csv(f'{DATA_DIR}common-stock.csv')
print(f"  Read {len(df_eq)} rows from common-stock.csv")

eq_prop_map = {
    'Sector': 'Sector', 'Exchange': 'Exchange', 'ListingCountry': 'ListingCountry',
    'DividendYield': 'DividendYield', 'MarketCapitalization': 'MarketCapitalization',
    'SharesOutstanding': 'SharesOutstanding', 'VotingRights': 'VotingRights',
    'EsgEnvironmentRating': 'EsgEnvironmentRating', 'EsgSocialRating': 'EsgSocialRating',
    'EsgGovernanceRating': 'EsgGovernanceRating', 'NaceLevel1': 'NaceLevel1',
    'CountryIso': 'CountryIso', 'DomicileCountry': 'DomicileCountry',
    'IssuerName': 'IssuerName', 'DataQualityStatus': 'DataQualityStatus',
    'DataOwner': 'DataOwner', 'WatchList': 'WatchList',
    'DataGovernanceCode': 'DataGovernanceCode', 'RiskProfile': 'RiskProfile',
}
eq_defs = {}
for idx, row in df_eq.iterrows():
    ci = str(row['ClientInternal']).strip()
    identifiers = {'ClientInternal': lm.InstrumentIdValue(value=ci)}
    for id_col in ['Isin', 'Figi', 'Sedol', 'Ticker']:
        v = str(row.get(id_col, '')).strip()
        if v and v != 'nan':
            identifiers[id_col] = lm.InstrumentIdValue(value=v)
    eq_defs[ci] = lm.InstrumentDefinition(
        name=str(row['CompanyName']).strip(),
        identifiers=identifiers,
        definition=lm.Equity(instrument_type='Equity', dom_ccy=str(row['Currency']).strip()),
        properties=list(build_properties(row, eq_prop_map, SCOPE).values()))
ok, fail = upsert_batch(eq_defs)
print(f"  Equities: {ok} succeeded, {fail} failed")

---
## Load Bonds

In [ ]:
print("--- Loading Bonds ---")
df_ig = pd.read_csv(f'{DATA_DIR}investment-grade.csv')
df_gov = pd.read_csv(f'{DATA_DIR}government.csv')
df_bonds = pd.concat([df_ig, df_gov], ignore_index=True)
print(f"  Read {len(df_bonds)} rows ({len(df_ig)} IG + {len(df_gov)} gov)")

bond_prop_map = {
    'BondType': 'BondType', 'Coupon': 'CouponRate', 'CouponFrequency': 'CouponFrequency',
    'CreditRating': 'CreditRating', 'DayCountConvention': 'DayCountConvention',
    'FaceValue': 'FaceValue', 'YieldToMaturity': 'YieldToMaturity',
    'CallableFlag': 'CallableFlag', 'PuttableFlag': 'PuttableFlag',
    'ConvertibleFlag': 'ConvertibleFlag', 'Subordination': 'Subordination',
    'Covenants': 'Covenants', 'AccruedInterest': 'AccruedInterest',
    'DefaultProbability': 'DefaultProbability', 'RecoveryRate': 'RecoveryRate',
    'CountryIso': 'CountryIso', 'IssuerName': 'IssuerName',
    'RatingS': 'RatingS', 'RatingM': 'RatingM', 'RatingF': 'RatingF',
    'DataQualityStatus': 'DataQualityStatus', 'DataOwner': 'DataOwner',
    'DataGovernanceCode': 'DataGovernanceCode',
}
bond_defs = {}
for idx, row in df_bonds.iterrows():
    ci = str(row['ClientInternal']).strip()
    identifiers = {'ClientInternal': lm.InstrumentIdValue(value=ci)}
    for id_col in ['Isin', 'Figi', 'Sedol']:
        v = str(row.get(id_col, '')).strip()
        if v and v != 'nan':
            identifiers[id_col] = lm.InstrumentIdValue(value=v)
    mat = str(row.get('MaturityDate', '2030-12-31')).strip()
    try:
        mat_dt = pd.to_datetime(mat).strftime('%Y-%m-%dT00:00:00Z')
    except:
        mat_dt = '2030-12-31T00:00:00Z'
    start = str(row.get('IssueDate', '2020-01-01')).strip()
    try:
        start_dt = pd.to_datetime(start).strftime('%Y-%m-%dT00:00:00Z')
    except:
        start_dt = '2020-01-01T00:00:00Z'
    coupon = float(row.get('Coupon', 0))
    bond_defs[ci] = lm.InstrumentDefinition(
        name=str(row['IssuerName']).strip(),
        identifiers=identifiers,
        definition=lm.Bond(
            instrument_type='Bond', dom_ccy=str(row['Currency']).strip(),
            start_date=start_dt, maturity_date=mat_dt,
            coupon_rate=coupon, principal=100,
            flow_conventions=lm.FlowConventions(
                currency=str(row['Currency']).strip(),
                payment_frequency='6M',
                day_count_convention='Thirty360',
                roll_convention='ModifiedFollowing',
                payment_calendars=[],
                reset_calendars=[])),
        properties=list(build_properties(row, bond_prop_map, SCOPE).values()))
ok, fail = upsert_batch(bond_defs)
print(f"  Bonds: {ok} succeeded, {fail} failed")

---
## Load ETFs, Options, FX Forwards, Private Investments

In [ ]:
# ETFs
print("--- Loading ETFs ---")
df_etf = pd.read_csv(f'{DATA_DIR}etfs.csv')
print(f"  Read {len(df_etf)} rows from etfs.csv")
etf_prop_map = {
    'Sector': 'Sector', 'AUM': 'AUM', 'ExpenseRatio': 'ExpenseRatio',
    'UnderlyingIndex': 'UnderlyingIndex', 'Replication': 'Replication',
    'CountryIso': 'CountryIso', 'EsgEnvironmentRating': 'EsgEnvironmentRating',
    'EsgSocialRating': 'EsgSocialRating', 'EsgGovernanceRating': 'EsgGovernanceRating',
    'DataQualityStatus': 'DataQualityStatus', 'DataOwner': 'DataOwner',
}
etf_defs = {}
for idx, row in df_etf.iterrows():
    ci = str(row['ClientInternal']).strip()
    identifiers = {'ClientInternal': lm.InstrumentIdValue(value=ci)}
    for id_col in ['Isin', 'Ticker', 'Sedol']:
        v = str(row.get(id_col, '')).strip()
        if v and v != 'nan':
            identifiers[id_col] = lm.InstrumentIdValue(value=v)
    etf_defs[ci] = lm.InstrumentDefinition(
        name=str(row['DisplayName']).strip(),
        identifiers=identifiers,
        definition=lm.Equity(instrument_type='Equity', dom_ccy=str(row['Currency']).strip()),
        properties=list(build_properties(row, etf_prop_map, SCOPE).values()))
ok, fail = upsert_batch(etf_defs)
print(f"  ETFs: {ok} succeeded, {fail} failed")

# Equity Options
print("\n--- Loading Equity Options ---")
df_opt = pd.read_csv(f'{DATA_DIR}equity-options.csv')
print(f"  Read {len(df_opt)} rows from equity-options.csv")
opt_defs = {}
for idx, row in df_opt.iterrows():
    ci = str(row['ClientInternal']).strip()
    exp = pd.to_datetime(str(row['OptionMaturityDate'])).strftime('%Y-%m-%dT00:00:00Z')
    start = pd.to_datetime(str(row['StartDate'])).strftime('%Y-%m-%dT00:00:00Z')
    opt_defs[ci] = lm.InstrumentDefinition(
        name=str(row['DisplayName']).strip(),
        identifiers={'ClientInternal': lm.InstrumentIdValue(value=ci)},
        definition=lm.EquityOption(
            instrument_type='EquityOption',
            start_date=start, option_maturity_date=exp,
            option_settlement_date=exp, delivery_type='Cash',
            strike=float(row['Strike']),
            dom_ccy=str(row.get('DomCcy', row.get('DomesticCurrency', 'USD'))).strip(),
            underlying_identifier='ClientInternal',
            code=str(row.get('Code', ci)).strip(),
            equity_option_type='Vanilla',
            option_type=str(row.get('OptionType', 'Call')).strip()))
ok, fail = upsert_batch(opt_defs)
print(f"  Options: {ok} succeeded, {fail} failed")

# FX Forwards
print("\n--- Loading FX Forwards ---")
df_fx = pd.read_csv(f'{DATA_DIR}fx-forwards.csv')
print(f"  Read {len(df_fx)} rows from fx-forwards.csv")
fx_defs = {}
for idx, row in df_fx.iterrows():
    ci = str(row['ClientInternal']).strip()
    mat = pd.to_datetime(str(row['MaturityDate'])).strftime('%Y-%m-%dT00:00:00Z')
    start = pd.to_datetime(str(row['StartDate'])).strftime('%Y-%m-%dT00:00:00Z')
    rate = float(row['Rate'])
    fx_defs[ci] = lm.InstrumentDefinition(
        name=f"{str(row['ClientInternal']).strip()}".strip(),
        identifiers={'ClientInternal': lm.InstrumentIdValue(value=ci)},
        definition=lm.FxForward(
            instrument_type='FxForward',
            dom_ccy=str(row['DomesticCurrency']).strip(), fgn_ccy=str(row['ForeignCurrency']).strip(),
            start_date=start, maturity_date=mat,
            dom_amount=1000000, fgn_amount=-1000000*rate, is_ndf=False))
ok, fail = upsert_batch(fx_defs)
print(f"  FX Forwards: {ok} succeeded, {fail} failed")

# Private Investments
print("\n--- Loading Private Investments ---")
df_pe = pd.read_csv(f'{DATA_DIR}growth-equity.csv')
print(f"  Read {len(df_pe)} rows from growth-equity.csv")
pe_defs = {}
for idx, row in df_pe.iterrows():
    ci = str(row['ClientInternal']).strip()
    pe_defs[ci] = lm.InstrumentDefinition(
        name=str(row['CompanyName']).strip(),
        identifiers={'ClientInternal': lm.InstrumentIdValue(value=ci)},
        definition=lm.SimpleInstrument(
            instrument_type='SimpleInstrument',
            dom_ccy=str(row.get('Currency', 'USD')).strip(),
            asset_class='Equities', simple_instrument_type='Other'))
ok, fail = upsert_batch(pe_defs)
print(f"  Private: {ok} succeeded, {fail} failed")

print("\n✅ NB02 Complete")